# Binary Model Review — Validation Template

Loads a paired **BUY** and **SELL** binary model pack, runs the full preprocessing pipeline on a held-out OHLCV dataset, applies trade-gating logic, and produces a detailed P&L breakdown.

**To use this notebook as a template:**
1. Update the model pack paths and dataset path in the **Configuration** cell
2. Adjust gating thresholds and fee settings as needed
3. Run all cells sequentially

| Stage | Description |
|---|---|
| Model Load | Unpacks BUY + SELL model packs; displays architecture & training metadata |
| Data Load | Loads validation-period OHLCV; adds features & scales using saved pipeline |
| Predictions | Batched softmax sweep — each bar produces `[sell_prob, hold_prob, buy_prob]` |
| Gating | Regime filter + breakout confirmation; incompatible signals masked to HOLD |
| P&L | Per-trade outcome simulation with fees; equity curve + drawdown |
| Insights | Threshold sweep, signal calibration, per-feature signal distribution |


In [ ]:
import os, pickle, types, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
warnings.filterwarnings('ignore')

In [ ]:
# ── MODEL PACKS ───────────────────────────────────────────────────────────────
BUY_PACK_PATH  = '../Engine/Model Packs/US500_1minute_LSTM_Binary_BUY_256seq_20260308_fastma_very_selective_wide_model.pkl'
SELL_PACK_PATH = '../Engine/Model Packs/US500_1minute_LSTM_Binary_SELL_256seq_20260308_fastma_very_selective_wide_model.pkl'

# ── VALIDATION DATASET ────────────────────────────────────────────────────────
DS_NAME        = '../data/US500_1minute.csv'   # OHLCV CSV to evaluate on
EVAL_START     = '2026-01-01'                   # Start of evaluation window (None = use full file)
EVAL_END       = None                           # End of evaluation window   (None = use full file)

# ── PREDICTION THRESHOLDS ─────────────────────────────────────────────────────
BUY_THRESHOLD  = 0.975   # Minimum buy_prob  to emit a BUY  signal
SELL_THRESHOLD = 0.975   # Minimum sell_prob to emit a SELL signal

# ── TRADE GATING ──────────────────────────────────────────────────────────────
GATE_REGIME    = True   # Mask signals that oppose the current market regime
GATE_BREAKOUT  = True   # Require the following bar to confirm the breakout
GATE_DONCHIAN  = True   # Require recent donchian trend to confirm signal direction (optional extra gate)
TRADING_HOURS  = None # ('07:00', '20:00')  # Anchor-time gate: signals outside this window are forced to
                                     # HOLD (0) before any other gating is applied.
                                     # Set to None to evaluate all hours.

# ── FEES & TRADE SIZING ───────────────────────────────────────────────────────
COMMISSION_USD  = 0.0    # Round-trip broker commission in USD per trade
LOT_SIZE        = 1.0    # Lot size per trade (0.1 = mini lot)
PIP_VALUE_USD   = 10.0   # USD value of 1 pip per standard lot (10.0 for EURUSD; 9.2 approx for GBPUSD etc.)
PIP_SIZE        = 0.0001 # Price change per pip (0.0001 for FX majors, 0.01 for JPY pairs)

# ── BATCH SIZE for prediction sweep ──────────────────────────────────────────
PRED_BATCH     = 2048

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


## Load Model Packs

Loads the BUY and SELL model packs and instantiates the model objects from the saved weights. If `Models.py` has changed since the pack was saved, the class is reconstructed from the embedded source snapshot.


In [ ]:
def _load_pack(path: str) -> dict:
    with open(path, 'rb') as f:
        return pickle.load(f)

def _build_model(pack: dict) -> torch.nn.Module:
    """Instantiate model from pack — falls back to embedded source if Models.py has changed."""
    model_params = pack['model_params']

    try:
        from Learn.Models import (
            TCNAttentionSEClassifier, LSTMAttentionSEClassifier, LSTMClassifier,
            TransformerClassifier, TransformerSEClassifier, HybridLSTMTransformer,
        )
        model_type = str(pack.get('model_info', {}).get('model_type', ''))
        if   'TCN'           in model_type: cls = TCNAttentionSEClassifier
        elif 'TransformerSE' in model_type: cls = TransformerSEClassifier
        elif 'Hybrid'        in model_type: cls = HybridLSTMTransformer
        elif 'Transformer'   in model_type: cls = TransformerClassifier
        elif 'LSTM'          in model_type: cls = LSTMAttentionSEClassifier
        else:                               cls = LSTMClassifier
        source = 'current Models.py'
    except Exception:
        # Reconstruct model class entirely from the saved source snapshot
        module = types.ModuleType('frozen_model')
        exec(pack['model_class_source'], module.__dict__)
        cls    = getattr(module, pack['model_class'].__name__)
        source = 'embedded source snapshot'

    model = cls(**model_params)
    model.load_state_dict(pack['model'])
    model.eval()
    return model, source

print("Loading model packs...")
buy_pack  = _load_pack(BUY_PACK_PATH)
sell_pack = _load_pack(SELL_PACK_PATH)

buy_model,  buy_source  = _build_model(buy_pack)
sell_model, sell_source = _build_model(sell_pack)

buy_model  = buy_model.to(device)
sell_model = sell_model.to(device)

print(f"  BUY  model loaded ({buy_source})  | params: {sum(p.numel() for p in buy_model.parameters()):,}")
print(f"  SELL model loaded ({sell_source}) | params: {sum(p.numel() for p in sell_model.parameters()):,}")


In [ ]:
def _display_pack_info(pack: dict, label: str):
    info    = pack.get('model_info', {})
    metrics = pack.get('val_metrics', {})
    params  = pack.get('model_params', {})
    split   = pack.get('data_split', {})

    rows = {
        '── Training ──────────────': '',
        'Dataset':         info.get('dataset_name', '—'),
        'Date trained':    info.get('date_trained', '—'),
        'Task':            f"{info.get('binary_task','—').upper()}  ({info.get('positive_class','—')} vs {info.get('negative_class','—')})",
        'Epochs':          info.get('n_epochs', '—'),
        'Train rows':      f"{split.get('train_rows', '—'):,}" if isinstance(split.get('train_rows'), int) else '—',
        'Val rows':        f"{split.get('val_rows', '—'):,}"   if isinstance(split.get('val_rows'),   int) else '—',
        'Val start':       split.get('test_start_date', '—'),
        'Torch version':   pack.get('torch_version', '—'),
        '── Architecture ──────────': '',
        'Seq len':         info.get('seq_len', '—'),
        'Input features':  pack.get('feature_count', '—'),
        'Input shape':     str(pack.get('input_shape', '—')),
        **{k: v for k, v in params.items()},
        '── Validation Metrics ────': '',
        'Final F1':        f"{metrics.get('final_f1', 0):.4f}"        if metrics else '—',
        'Best F1':         f"{metrics.get('best_f1', 0):.4f}"         if metrics else '—',
        'Final Precision': f"{metrics.get('final_precision', 0):.4f}" if metrics else '—',
        'Final Recall':    f"{metrics.get('final_recall', 0):.4f}"    if metrics else '—',
        'Final Val Profit':f"{metrics.get('final_profit', 0):.2f}"    if metrics else '—',
    }

    df_info = pd.DataFrame({'Field': rows.keys(), 'Value': rows.values()})
    print(f"\n{'═'*50}")
    print(f"  {label}")
    print(f"{'═'*50}")
    for _, row in df_info.iterrows():
        sep = '─' in str(row['Field'])
        if sep:
            print(f"  {row['Field']}")
        else:
            print(f"  {row['Field']:<22} {row['Value']}")

_display_pack_info(buy_pack,  '🔼 BUY MODEL')
_display_pack_info(sell_pack, '🔽 SELL MODEL')


In [ ]:
# Plot training metric curves from both packs side-by-side
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Training Curves — BUY (blue) vs SELL (orange)', fontsize=13)

for pack, color, lbl in [(buy_pack, 'tab:blue', 'BUY'), (sell_pack, 'tab:orange', 'SELL')]:
    m = pack.get('val_metrics', {})
    if not m:
        continue
    axes[0, 0].plot(m.get('f1_curve', []),        color=color, label=lbl)
    axes[0, 1].plot(m.get('precision_curve', []), color=color, label=lbl)
    axes[1, 0].plot(m.get('recall_curve', []),    color=color, label=lbl)
    axes[1, 1].plot(m.get('pnl_curve', []),       color=color, label=lbl)

for ax, title in zip(axes.flat, ['Val F1', 'Val Precision', 'Val Recall', 'Val Profit']):
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.show()


## Load Validation Dataset

Loads the raw OHLCV CSV clipped to the evaluation window. The model packs contain all feature-engineering and preprocessing functions so no manual configuration is required here.


In [ ]:
df_raw = pd.read_csv(DS_NAME)
df_raw = df_raw.sort_values('Time').reset_index(drop=True)
df_raw['Time'] = pd.to_datetime(df_raw['Time'])

if EVAL_START:
    df_raw = df_raw[df_raw['Time'] >= EVAL_START]
if EVAL_END:
    df_raw = df_raw[df_raw['Time'] <= EVAL_END]

df_raw = df_raw.reset_index(drop=True)
print(f"Evaluation window: {df_raw['Time'].iloc[0]}  →  {df_raw['Time'].iloc[-1]}")
print(f"Total bars: {len(df_raw):,}")
df_raw.tail(3)


## Features & Preprocessing

Applies the exact same feature engineering and scaling pipeline that was used during training — loaded directly from each model pack. The BUY and SELL packs may use different scalers so both are applied independently.


In [ ]:
def _prepare_features(df: pd.DataFrame, pack: dict) -> np.ndarray:
    """Add features and scale using the saved pipeline from a model pack.
    Returns the scaled feature array aligned to df (NaN rows dropped)."""
    feature_fn   = pack['feature_function']
    preprocess_fn = pack['preprocess_function']
    preprocess_args = {**pack['preprocess_args'], 'target_col': None, 'outcomes_col': None}
    scaler = pack['scaler']

    df_feat = feature_fn(df.copy(), regime_params=pack.get('regime_params'))
    df_feat = df_feat.dropna().reset_index(drop=True)

    X, _, _, _, _, df_out = preprocess_fn(df_feat, scaler=scaler, return_df=True, **preprocess_args)
    X = np.ascontiguousarray(X)
    return X, df_feat, df_out

print("Building buy feature matrix...")
X_buy, df_buy, _ = _prepare_features(df_raw, buy_pack)
print(f"  BUY  matrix: {X_buy.shape}")

print("Building sell feature matrix...")
X_sell, df_sell, _ = _prepare_features(df_raw, sell_pack)
print(f"  SELL matrix: {X_sell.shape}")


## Batch Predictions

Sweeps the full evaluation period using a sliding window equal to `seq_len`. Each bar receives a **3-value probability vector**: `[sell_prob, hold_prob, buy_prob]`.

- `buy_prob`  = softmax output class 1 from the **BUY model**  
- `sell_prob` = softmax output class 1 from the **SELL model**  
- `hold_prob` = `1 − buy_prob − sell_prob` (clamped to 0)

The raw signal is the `argmax`: **−1** = SELL, **0** = HOLD, **1** = BUY — thresholded by `BUY_THRESHOLD` / `SELL_THRESHOLD`.


In [ ]:
def _batch_predict(model: torch.nn.Module, X: np.ndarray, seq_len: int, batch_size: int = 2048):
    """Sliding-window batch inference. Returns prob_trade array (class 1) for each valid timestep."""
    N = len(X)
    n_windows = N - seq_len
    if n_windows <= 0:
        raise ValueError(f"Feature matrix too short ({N} rows) for seq_len={seq_len}")

    # Stack all windows: shape [n_windows, seq_len, features]
    idx = np.arange(n_windows)
    seqs = np.stack([X[i:i + seq_len] for i in idx])          # (n_windows, seq_len, F)

    prob_trade = np.zeros(n_windows, dtype=np.float32)

    model.eval()
    with torch.no_grad():
        for start in range(0, n_windows, batch_size):
            batch = torch.tensor(seqs[start:start + batch_size], dtype=torch.float32).to(device)
            logits = model(batch)
            probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            prob_trade[start:start + len(probs)] = probs

    return prob_trade   # length = N - seq_len, aligned to df_feat.iloc[seq_len:]

buy_seq_len  = int(buy_pack['model_info']['seq_len'])
sell_seq_len = int(sell_pack['model_info']['seq_len'])

print(f"Running BUY  model  (seq_len={buy_seq_len}, {len(X_buy):,} rows) ...")
buy_probs  = _batch_predict(buy_model,  X_buy,  buy_seq_len,  PRED_BATCH)

print(f"Running SELL model  (seq_len={sell_seq_len}, {len(X_sell):,} rows) ...")
sell_probs = _batch_predict(sell_model, X_sell, sell_seq_len, PRED_BATCH)

# ── Align both prediction series to a shared Time index ──────────────────────
# Each model's predictions start at seq_len-th row of its respective df_feat
buy_times  = df_buy['Time'].iloc[buy_seq_len:].reset_index(drop=True)
sell_times = df_sell['Time'].iloc[sell_seq_len:].reset_index(drop=True)

buy_pred_df  = pd.DataFrame({'Time': buy_times,  'buy_prob':  buy_probs})
sell_pred_df = pd.DataFrame({'Time': sell_times, 'sell_prob': sell_probs})

preds = pd.merge(buy_pred_df, sell_pred_df, on='Time', how='inner')
preds['hold_prob'] = np.clip(1.0 - preds['buy_prob'] - preds['sell_prob'], 0.0, 1.0)

# ── Raw signals using configured thresholds ───────────────────────────────────
preds['raw_signal'] = 0
preds.loc[preds['buy_prob']  >= BUY_THRESHOLD,  'raw_signal'] =  1
preds.loc[preds['sell_prob'] >= SELL_THRESHOLD, 'raw_signal'] = -1
# When both fire, prefer higher probability
both = (preds['buy_prob'] >= BUY_THRESHOLD) & (preds['sell_prob'] >= SELL_THRESHOLD)
preds.loc[both & (preds['buy_prob'] >= preds['sell_prob']), 'raw_signal'] =  1
preds.loc[both & (preds['sell_prob'] >  preds['buy_prob']), 'raw_signal'] = -1

# ── Anchor-time session gate ──────────────────────────────────────────────────
# Signals whose prediction timestamp falls outside TRADING_HOURS are forced to
# HOLD before any downstream gating (regime, breakout, etc.) is applied.
# This mirrors the anchor-time filtering used during model training.
if TRADING_HOURS is not None:
    t_start    = pd.to_datetime(TRADING_HOURS[0]).time()
    t_end      = pd.to_datetime(TRADING_HOURS[1]).time()
    in_session = preds['Time'].dt.time.between(t_start, t_end, inclusive='left')
    n_masked   = (~in_session).sum()
    preds.loc[~in_session, 'raw_signal'] = 0
    print(f"Session gate ({TRADING_HOURS}): {n_masked:,} bars forced to HOLD "
          f"({n_masked / len(preds) * 100:.1f}% of prediction window)")

print(f"\nPrediction rows: {len(preds):,}")
print("Raw signal distribution:")
print(preds['raw_signal'].value_counts().rename({-1:'SELL', 0:'HOLD', 1:'BUY'}))


In [ ]:

# ── Signal Confidence Distribution (interactive) ──────────────────────────────
from plotly.subplots import make_subplots
import plotly.graph_objects as go

n_buy  = (preds['raw_signal'] ==  1).sum()
n_sell = (preds['raw_signal'] == -1).sum()
n_hold = (preds['raw_signal'] ==  0).sum()

bins   = dict(start=0, end=1, size=0.025)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        f'BUY Model — P(trade)  [signals: {n_buy:,}]',
        f'SELL Model — P(trade)  [signals: {n_sell:,}]',
        f'BUY vs SELL  [B:{n_buy:,}  S:{n_sell:,}  H:{n_hold:,}]',
    ),
    horizontal_spacing=0.08,
)

# ── Panel 1: BUY histogram ────────────────────────────────────────────────────
fig.add_trace(go.Histogram(
    x=preds['buy_prob'], xbins=bins,
    marker_color='steelblue', opacity=0.85, name='BUY prob',
    showlegend=False,
), row=1, col=1)
fig.add_vline(x=BUY_THRESHOLD,              line=dict(color='red',    dash='dash', width=1.5),
              annotation_text=f'Thresh {BUY_THRESHOLD}', annotation_position='top right', row=1, col=1)
fig.add_vline(x=float(preds['buy_prob'].mean()), line=dict(color='orange', dash='dot',  width=1.5),
              annotation_text=f'Mean {preds["buy_prob"].mean():.2f}',  annotation_position='top left',  row=1, col=1)

# ── Panel 2: SELL histogram ───────────────────────────────────────────────────
fig.add_trace(go.Histogram(
    x=preds['sell_prob'], xbins=bins,
    marker_color='tomato', opacity=0.85, name='SELL prob',
    showlegend=False,
), row=1, col=2)
fig.add_vline(x=SELL_THRESHOLD,              line=dict(color='red',    dash='dash', width=1.5),
              annotation_text=f'Thresh {SELL_THRESHOLD}', annotation_position='top right', row=1, col=2)
fig.add_vline(x=float(preds['sell_prob'].mean()), line=dict(color='orange', dash='dot',  width=1.5),
              annotation_text=f'Mean {preds["sell_prob"].mean():.2f}',  annotation_position='top left',  row=1, col=2)

# ── Panel 3: BUY vs SELL scatter ──────────────────────────────────────────────
colour_map  = {1: 'steelblue', -1: 'tomato', 0: 'lightgrey'}
label_map   = {1: 'BUY',       -1: 'SELL',   0: 'HOLD'}

for sig_val in [0, 1, -1]:           # draw HOLD first so signals sit on top
    mask = preds['raw_signal'] == sig_val
    fig.add_trace(go.Scattergl(
        x=preds.loc[mask, 'buy_prob'],
        y=preds.loc[mask, 'sell_prob'],
        mode='markers',
        marker=dict(color=colour_map[sig_val], size=3, opacity=0.35),
        name=label_map[sig_val],
    ), row=1, col=3)

# Threshold crosshairs
fig.add_vline(x=BUY_THRESHOLD,  line=dict(color='steelblue', dash='dash', width=1), row=1, col=3)
fig.add_hline(y=SELL_THRESHOLD, line=dict(color='tomato',    dash='dash', width=1), row=1, col=3)

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    title_text='Signal Confidence Distribution (all prediction bars)',
    title_font_size=14,
    height=420,
    legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='right', x=1),
    bargap=0.05,
)
fig.update_xaxes(title_text='Probability', row=1, col=1)
fig.update_xaxes(title_text='Probability', row=1, col=2)
fig.update_xaxes(title_text='BUY P(trade)', row=1, col=3)
fig.update_yaxes(title_text='Bars',         row=1, col=1)
fig.update_yaxes(title_text='Bars',         row=1, col=2)
fig.update_yaxes(title_text='SELL P(trade)',row=1, col=3)

fig.show()


## Trade Gating

Two gating conditions are applied sequentially. Signals that fail any active gate are changed to **HOLD (0)**.

| Gate | Condition |
|---|---|
| **Regime** | BUY signals only allowed during uptrend (`regime == 1`); SELL signals only during downtrend (`regime == -1`) |
| **Breakout** | BUY confirmed if the *following bar's* high breaks above the signal bar's high; SELL confirmed if the following bar's low breaks below the signal bar's low |

Gates are individually toggled via `GATE_REGIME` and `GATE_BREAKOUT` in the configuration cell.


In [ ]:
from Learn.labels import causal_market_regime
from Learn.features import donchian_trend

# ── Compute regime over the full eval window ──────────────────────────────────
# Use regime_params from buy_pack (both packs trained with same regime config)
regime_params = buy_pack.get('regime_params', {})
regime_df = causal_market_regime(df_raw.copy(), **regime_params)
df_raw['regime'] = regime_df.values
df_raw['donchian_trend'] = donchian_trend(df_raw, length=regime_params['atr_window'])

# ── Merge preds with raw OHLC (for High/Low breakout check) and regime ────────
ohlc_cols = df_raw[['Time', 'Open', 'High', 'Low', 'Close', 'regime', 'donchian_trend']].copy()
df_gated = pd.merge(preds, ohlc_cols, on='Time', how='left')

# Forward-shift OHLC by 1 to get next bar's prices (breakout confirmation)
df_ohlc_aligned = df_raw[['Time', 'High', 'Low']].rename(
    columns={'High': 'next_high', 'Low': 'next_low'})
df_ohlc_aligned['Time'] = df_ohlc_aligned['Time'].shift(1)   # shift Times forward to match signal bars
df_gated = pd.merge(df_gated, df_ohlc_aligned, on='Time', how='left')

df_gated['signal'] = df_gated['raw_signal'].copy()

# ── Gate 1: Regime ────────────────────────────────────────────────────────────
if GATE_REGIME:
    # Mask BUY signals in downtrend (regime == -1)
    regime_buy_block  = (df_gated['signal'] ==  1) & (df_gated['regime'] != 1)
    # Mask SELL signals in uptrend (regime == 1)
    regime_sell_block = (df_gated['signal'] == -1) & (df_gated['regime'] != -1)
    df_gated.loc[regime_buy_block | regime_sell_block, 'signal'] = 0
    print(f"Regime gate removed {(regime_buy_block | regime_sell_block).sum():,} signals")

# ── Gate 2: Breakout confirmation ─────────────────────────────────────────────
if GATE_BREAKOUT:
    # BUY confirmed only if next bar's high > signal bar's high
    buy_no_breakout  = (df_gated['signal'] ==  1) & (df_gated['next_high'] <= df_gated['High'])
    # SELL confirmed only if next bar's low < signal bar's low
    sell_no_breakout = (df_gated['signal'] == -1) & (df_gated['next_low']  >= df_gated['Low'])
    df_gated.loc[buy_no_breakout | sell_no_breakout, 'signal'] = 0
    print(f"Breakout gate removed {(buy_no_breakout | sell_no_breakout).sum():,} signals")

# Gate 3: Donchian trend (optional extra gate based on recent channel direction)
if GATE_DONCHIAN:
    # BUY confirmed only if donchian trend is positive
    buy_no_donchian  = (df_gated['signal'] ==  1) & (df_gated['donchian_trend'] <= 0)
    # SELL confirmed only if donchian trend is negative
    sell_no_donchian = (df_gated['signal'] == -1) & (df_gated['donchian_trend'] >= 0)
    df_gated.loc[buy_no_donchian | sell_no_donchian, 'signal'] = 0
    print(f"Donchian gate removed {(buy_no_donchian | sell_no_donchian).sum():,} signals")

# ── Summary ───────────────────────────────────────────────────────────────────
raw_counts  = df_gated['raw_signal'].value_counts().rename({-1:'SELL', 0:'HOLD', 1:'BUY'})
gate_counts = df_gated['signal'].value_counts().rename({-1:'SELL', 0:'HOLD', 1:'BUY'})

summary = pd.DataFrame({'Raw': raw_counts, 'Gated': gate_counts}).fillna(0).astype(int)
summary['Removed'] = summary['Raw'] - summary['Gated']
print(f"\n{summary.to_string()}")


## Candlestick Chart

Interactive OHLCV candlestick chart overlaid with gated trade signals.

- **▲ Green triangle (below bar)** — BUY signal (after regime + breakout gates)
- **▼ Red triangle (above bar)** — SELL signal (after regime + breakout gates)

Use the range-selector buttons or drag the range-slider at the bottom to zoom into any period. Hover over a marker to see the bar time and probability score.


In [ ]:
## Visualise signals on candlestick chart with Plotly, showing how gating affects signal placement and frequency.
# ── Candlestick chart with gated signal markers ───────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Merge signal data with OHLC so markers have price context
df_chart = df_raw[['Time', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

# Pull gated signals and signal-bar probabilities
sig_cols = ['Time', 'signal', 'buy_prob', 'sell_prob']
df_chart = df_chart.merge(df_gated[sig_cols], on='Time', how='left')
df_chart['signal'] = df_chart['signal'].fillna(0).astype(int)

buy_bars  = df_chart[df_chart['signal'] ==  1]
sell_bars = df_chart[df_chart['signal'] == -1]

# ATR-based marker offset so arrows sit clear of the wicks
atr_series = df_chart['High'].rolling(14).mean() - df_chart['Low'].rolling(14).mean()
atr_mean   = float(atr_series.mean())
offset     = atr_mean * 0.5

# Volume bar colours: green if close >= open, red otherwise
vol_colors = np.where(df_chart['Close'] >= df_chart['Open'], '#26a69a', '#ef5350')

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.8, 0.2],
    vertical_spacing=0.02,
)

# ── Candlesticks ──────────────────────────────────────────────────────────────
fig.add_trace(go.Candlestick(
    x=df_chart['Time'],
    open=df_chart['Open'], high=df_chart['High'],
    low=df_chart['Low'],   close=df_chart['Close'],
    name='OHLCV',
    increasing_line_color='#26a69a',
    decreasing_line_color='#ef5350',
    increasing_fillcolor='#26a69a',
    decreasing_fillcolor='#ef5350',
    line=dict(width=1),
    hoverinfo='skip',
), row=1, col=1)

# ── BUY markers — triangles below the low ────────────────────────────────────
fig.add_trace(go.Scatter(
    x=buy_bars['Time'],
    y=buy_bars['Low'] - offset,
    mode='markers',
    marker=dict(symbol='triangle-up', size=9, color='#00e676',
                line=dict(color='darkgreen', width=0.8)),
    name='BUY signal',
    hoverinfo='skip',
), row=1, col=1)

# ── SELL markers — triangles above the high ───────────────────────────────────
fig.add_trace(go.Scatter(
    x=sell_bars['Time'],
    y=sell_bars['High'] + offset,
    mode='markers',
    marker=dict(symbol='triangle-down', size=9, color='#ff1744',
                line=dict(color='darkred', width=0.8)),
    name='SELL signal',
    hoverinfo='skip',
), row=1, col=1)

# ── Volume bars ───────────────────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=df_chart['Time'],
    y=df_chart['Volume'],
    marker_color=vol_colors,
    marker_line_width=0,
    name='Volume',
    hoverinfo='skip',
    showlegend=False,
), row=2, col=1)

# ── Layout ────────────────────────────────────────────────────────────────────
gate_label = []
if GATE_REGIME:    gate_label.append('regime')
if GATE_BREAKOUT:  gate_label.append('breakout')
gate_str = ' + '.join(gate_label) if gate_label else 'none'

fig.update_layout(
    xaxis=dict(
        rangeslider=dict(visible=False),
        rangeselector=dict(buttons=[
            dict(count=1,  label='1d',  step='day',   stepmode='backward'),
            dict(count=7,  label='1w',  step='day',   stepmode='backward'),
            dict(count=1,  label='1mo', step='month', stepmode='backward'),
            dict(count=3,  label='3mo', step='month', stepmode='backward'),
            dict(step='all', label='All'),
        ]),
        type='date',
    ),
    xaxis2=dict(title='Time'),
    yaxis=dict(title='Price', fixedrange=False),
    yaxis2=dict(title='Volume', fixedrange=True),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=1200,
    margin=dict(l=60, r=20, t=80, b=60),
    hovermode=False,
    paper_bgcolor='#1e1e1e',
    plot_bgcolor='#1e1e1e',
    font=dict(color='#d4d4d4'),
    xaxis_gridcolor='#333',
    yaxis_gridcolor='#333',
    xaxis2_gridcolor='#333',
    yaxis2_gridcolor='#333',
)

fig.show()


## P&L Analysis

Simulates each gated trade signal against actual forward price action using the ATR-based TP/SL configuration from the model pack's `outcome_params`. Fees are deducted from every trade.

**Entry logic mirrors the live engine:**
- Entry price = signal bar's High (BUY) or Low (SELL) — stop-entry order fills on the following bar  
- TP/SL distances are ATR × `tp_mult` / `sl_mult` from entry  
- Outcome: **+TP** (hit take-profit), **−SL** (hit stop-loss), **timeout** (expired after `max_horizon` bars)


In [ ]:
from Learn.labels import calculate_trade_outcomes_all_candles

outcome_params = buy_pack.get('outcome_params', {})
atr_window  = outcome_params.get('atr_window', 14)
tp_mult     = outcome_params.get('tp_mult', 4)
sl_mult     = outcome_params.get('sl_mult', 2)
max_horizon = outcome_params.get('max_horizon', 60)

pip_value_per_lot = PIP_VALUE_USD * LOT_SIZE   # USD per pip for the configured lot size

print(f"Outcome params — ATR window: {atr_window} | TP: {tp_mult}×ATR | SL: {sl_mult}×ATR | Horizon: {max_horizon} bars")
print(f"Fee: ${COMMISSION_USD:.2f} round-trip commission | Lot size: {LOT_SIZE} | Pip value: ${pip_value_per_lot:.2f}/pip")

# ── Precompute outcomes for ALL candles using the same logic as training ───────
df_sim = df_raw.copy().reset_index(drop=True)

print("Computing trade outcomes for all candles...")
outcomes_table = calculate_trade_outcomes_all_candles(
    df_sim,
    atr_window=atr_window,
    tp_mult=tp_mult,
    sl_mult=sl_mult,
    max_horizon=max_horizon,
)
df_sim['buy_outcome']    = outcomes_table['buy_outcome'].values
df_sim['sell_outcome']   = outcomes_table['sell_outcome'].values
df_sim['buy_exit_price'] = outcomes_table['buy_exit_price'].values
df_sim['sell_exit_price']= outcomes_table['sell_exit_price'].values

# Recompute ATR for entry/TP/SL price display (already computed inside the function but not exposed)
import talib
df_sim['atr'] = talib.ATR(df_sim['High'], df_sim['Low'], df_sim['Close'], timeperiod=atr_window)

# Build a Time→index lookup for fast lookup
time_to_idx = {t: i for i, t in enumerate(df_sim['Time'])}

def _simulate_trade(signal_time, direction):
    """Look up precomputed outcome for the signal bar.
    direction: +1 = BUY, -1 = SELL
    Returns dict with trade results."""
    idx = time_to_idx.get(signal_time)
    if idx is None:
        return None

    row = df_sim.iloc[idx]
    atr = row['atr']
    if pd.isna(atr):
        return None

    if direction == 1:   # BUY
        entry      = row['High']
        tp         = entry + atr * tp_mult
        sl         = entry - atr * sl_mult
        raw_outcome= row['buy_outcome']
        exit_price = row['buy_exit_price']
    else:                # SELL
        entry      = row['Low']
        tp         = entry - atr * tp_mult
        sl         = entry + atr * sl_mult
        raw_outcome= row['sell_outcome']
        exit_price = row['sell_exit_price']

    if pd.isna(raw_outcome) or pd.isna(exit_price):
        return None

    # Classify outcome: exactly 1 = TP, exactly -1 = SL, fractional = timeout
    if raw_outcome == 1.0:
        outcome = 'tp'
    elif raw_outcome == -1.0:
        outcome = 'sl'
    else:
        outcome = 'timeout'

    raw_pips = direction * (exit_price - entry) / PIP_SIZE
    raw_usd  = raw_pips * pip_value_per_lot
    net_usd  = raw_usd - COMMISSION_USD

    return {
        'time':       signal_time,
        'direction':  'BUY' if direction == 1 else 'SELL',
        'entry':      round(entry, 5),
        'tp':         round(tp, 5),
        'sl':         round(sl, 5),
        'exit_price': round(exit_price, 5),
        'atr_pips':   round(atr / PIP_SIZE, 2),
        'outcome':    outcome,
        'raw_pips':   round(raw_pips, 2),
        'raw_usd':    round(raw_usd, 2),
        'net_usd':    round(net_usd, 2),
        'buy_prob':   float(df_gated.loc[df_gated['Time'] == signal_time, 'buy_prob'].iloc[0]) if direction == 1 else None,
        'sell_prob':  float(df_gated.loc[df_gated['Time'] == signal_time, 'sell_prob'].iloc[0]) if direction == -1 else None,
    }

# ── Run simulation for all gated signals ─────────────────────────────────────
trades_raw = []
signal_bars = df_gated[df_gated['signal'] != 0]
for _, row in signal_bars.iterrows():
    result = _simulate_trade(row['Time'], int(row['signal']))
    if result:
        trades_raw.append(result)

trades = pd.DataFrame(trades_raw)
print(f"\n{'─'*50}")
print(f"  Total trades simulated: {len(trades):,}")
if len(trades):
    tp_count  = (trades['outcome'] == 'tp').sum()
    sl_count  = (trades['outcome'] == 'sl').sum()
    to_count  = (trades['outcome'] == 'timeout').sum()
    win_rate  = tp_count / len(trades) * 100
    net_total = trades['net_usd'].sum()
    print(f"  TP: {tp_count} | SL: {sl_count} | Timeout: {to_count}")
    print(f"  Win rate:    {win_rate:.1f}%")
    print(f"  Net P&L:     ${net_total:+,.2f}")
    print(f"  Avg / trade: ${trades['net_usd'].mean():+.2f}")


In [ ]:
if len(trades) == 0:
    print("No trades to display.")
else:
    # ── Per-direction breakdown ────────────────────────────────────────────────
    def _summary_stats(df):
        if len(df) == 0:
            return {}
        return {
            'Trades':          len(df),
            'TP':              (df['outcome'] == 'tp').sum(),
            'SL':              (df['outcome'] == 'sl').sum(),
            'Timeout':         (df['outcome'] == 'timeout').sum(),
            'Win Rate %':      round((df['outcome'] == 'tp').mean() * 100, 1),
            'Net USD ($)':     round(df['net_usd'].sum(), 2),
            'Avg USD ($)':     round(df['net_usd'].mean(), 2),
            'Best Trade ($)':  round(df['net_usd'].max(), 2),
            'Worst Trade ($)': round(df['net_usd'].min(), 2),
            'Commission ($)':  round(COMMISSION_USD * len(df), 2),
        }

    stats = pd.DataFrame({
        'ALL':  _summary_stats(trades),
        'BUY':  _summary_stats(trades[trades['direction'] == 'BUY']),
        'SELL': _summary_stats(trades[trades['direction'] == 'SELL']),
    }).T
    display(stats)

    # ── Equity curve ──────────────────────────────────────────────────────────
    trades_sorted = trades.sort_values('time').reset_index(drop=True)
    trades_sorted['cum_usd'] = trades_sorted['net_usd'].cumsum()

    running_max = trades_sorted['cum_usd'].cummax()
    drawdown    = trades_sorted['cum_usd'] - running_max
    max_dd      = drawdown.min()

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1.5, 1]})

    # Equity
    axes[0].plot(trades_sorted.index, trades_sorted['cum_usd'], color='steelblue', lw=1.5, label='Equity')
    axes[0].fill_between(trades_sorted.index, trades_sorted['cum_usd'], alpha=0.15, color='steelblue')
    axes[0].axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    axes[0].set_title(f'Cumulative Net P&L  ({len(trades):,} trades | ${trades_sorted["cum_usd"].iloc[-1]:+,.2f} | MaxDD: ${max_dd:.2f})')
    axes[0].set_ylabel('USD ($)'); axes[0].grid(alpha=0.3); axes[0].legend()

    # Per-trade bar chart coloured by outcome
    colours = trades_sorted['outcome'].map({'tp': 'seagreen', 'sl': 'firebrick', 'timeout': 'goldenrod'})
    axes[1].bar(trades_sorted.index, trades_sorted['net_usd'], color=colours, width=0.8)
    axes[1].axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    axes[1].set_ylabel('USD ($)'); axes[1].set_title('Per-Trade P&L  (green=TP, red=SL, gold=timeout)'); axes[1].grid(alpha=0.3)

    # Drawdown
    axes[2].fill_between(trades_sorted.index, drawdown, 0, color='firebrick', alpha=0.4)
    axes[2].set_ylabel('DD ($)'); axes[2].set_title('Drawdown'); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


## Model Insights

### Threshold Sweep
Sweeps both signal thresholds from 0.3→0.9 and measures the impact on **trade count, win rate, and total net pips**. Useful for calibrating thresholds after training.

### Probability Calibration
Histograms of model output probabilities split by actual trade outcome (TP / SL / timeout). Well-calibrated models should show high-probability signals correlating strongly with TP outcomes.

### Gating Effectiveness
Compares raw vs regime-gated vs fully-gated win rates to quantify the value of each gate.


In [ ]:
if len(trades) == 0:
    print("No trades available — run simulation cell first.")
else:
    thresholds = np.arange(0.30, 0.90, 0.05)
    sweep_rows = []

    for thresh in thresholds:
        # Re-derive signals at this threshold
        sig = np.zeros(len(df_gated), dtype=int)
        sig[df_gated['buy_prob'].values  >= thresh] =  1
        sig[df_gated['sell_prob'].values >= thresh] = -1
        both_mask = (df_gated['buy_prob'].values >= thresh) & (df_gated['sell_prob'].values >= thresh)
        sig[both_mask & (df_gated['buy_prob'].values >= df_gated['sell_prob'].values)] =  1
        sig[both_mask & (df_gated['sell_prob'].values >  df_gated['buy_prob'].values)] = -1

        # Apply regime gate (if enabled)
        if GATE_REGIME:
            sig[(sig ==  1) & (df_gated['regime'].values != 1)]  = 0
            sig[(sig == -1) & (df_gated['regime'].values != -1)] = 0

        # Apply breakout gate (if enabled)
        if GATE_BREAKOUT:
            sig[(sig ==  1) & (df_gated['next_high'].values <= df_gated['High'].values)] = 0
            sig[(sig == -1) & (df_gated['next_low'].values  >= df_gated['Low'].values)]  = 0

        signal_times = df_gated['Time'].values[sig != 0]
        directions   = sig[sig != 0]

        t_list = []
        for st, d in zip(signal_times, directions):
            r = _simulate_trade(pd.Timestamp(st), int(d))
            if r:
                t_list.append(r)

        if not t_list:
            sweep_rows.append({'threshold': round(thresh, 2), 'trades': 0,
                               'win_pct': 0, 'net_usd': 0, 'avg_usd': 0})
            continue

        t_df = pd.DataFrame(t_list)
        sweep_rows.append({
            'threshold': round(thresh, 2),
            'trades':    len(t_df),
            'win_pct':   round((t_df['outcome'] == 'tp').mean() * 100, 1),
            'net_usd':   round(t_df['net_usd'].sum(), 2),
            'avg_usd':   round(t_df['net_usd'].mean(), 2),
        })

    sweep_df = pd.DataFrame(sweep_rows)
    display(sweep_df.set_index('threshold'))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(sweep_df['threshold'], sweep_df['trades'],   marker='o', color='steelblue')
    axes[0].set_title('Trade Count vs Threshold'); axes[0].set_xlabel('Threshold'); axes[0].grid(alpha=0.3)

    axes[1].plot(sweep_df['threshold'], sweep_df['win_pct'],  marker='o', color='seagreen')
    axes[1].axhline(50, color='red', ls='--', alpha=0.5, label='50%')
    axes[1].set_title('Win Rate % vs Threshold'); axes[1].set_xlabel('Threshold')
    axes[1].set_ylabel('%'); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(sweep_df['threshold'], sweep_df['net_usd'], marker='o', color='darkorange')
    axes[2].axhline(0, color='black', ls='--', alpha=0.5)
    axes[2].set_title('Net USD ($) vs Threshold'); axes[2].set_xlabel('Threshold')
    axes[2].set_ylabel('USD ($)'); axes[2].grid(alpha=0.3)

    plt.suptitle('Threshold Sweep', fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Probability Calibration ───────────────────────────────────────────────────
if len(trades) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, col, title in [
        (axes[0], 'buy_prob',  'BUY Model — Probability by Outcome'),
        (axes[1], 'sell_prob', 'SELL Model — Probability by Outcome'),
    ]:
        subset = trades[trades[col].notna()]
        if subset.empty:
            ax.set_visible(False); continue
        for outcome, color in [('tp', 'seagreen'), ('sl', 'firebrick'), ('timeout', 'goldenrod')]:
            vals = subset.loc[subset['outcome'] == outcome, col].dropna()
            if len(vals):
                ax.hist(vals, bins=20, alpha=0.55, color=color, label=f'{outcome} (n={len(vals)})',
                        density=True)
        ax.set_title(title); ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Density')
        ax.legend(); ax.grid(alpha=0.3)

    plt.suptitle('Calibration — are high-confidence signals more likely to TP?', fontsize=12)
    plt.tight_layout(); plt.show()

# ── Gating Effectiveness ──────────────────────────────────────────────────────
print("\nGating Effectiveness (win rate %):")

def _win_rate_for_signals(sig_series):
    results = []
    for _, row in df_gated[sig_series != 0].iterrows():
        r = _simulate_trade(row['Time'], int(sig_series.loc[row.name]))
        if r:
            results.append(r)
    if not results:
        return 0, 0
    t = pd.DataFrame(results)
    return len(t), round((t['outcome'] == 'tp').mean() * 100, 1)

raw_sig   = df_gated['raw_signal']
gated_sig = df_gated['signal']

n_raw,   wr_raw   = _win_rate_for_signals(raw_sig)
n_gated, wr_gated = _win_rate_for_signals(gated_sig)

gate_summary = pd.DataFrame({
    'Scenario':  ['Raw (no gating)', 'Fully gated'],
    'Trades':    [n_raw,   n_gated],
    'Win Rate %':[wr_raw,  wr_gated],
})
display(gate_summary.set_index('Scenario'))
